In [2]:
from pathlib import Path
import random
import numpy as np
import torch

# 固定随机种子，与 CNN 基线保持一致
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# 加载已经保存、已经标准化的数据
files = list(Path("/kaggle/input").rglob("metadata.npz"))
assert len(files) == 1, f"找到 {len(files)} 个元数据文件，请检查输入"

prepared_dir = files[0].parent

X = np.load(prepared_dir / "X.npy", mmap_mode="r")
y = np.load(prepared_dir / "y.npy")
info = np.load(prepared_dir / "metadata.npz")

assert X.shape == (len(y), 12, 1000)
assert y.shape[1] == len(info["classes"])

print("波形：", X.shape)
print("标签：", y.shape)
print("类别：", info["classes"])
print("训练记录：", info["train_mask"].sum())
print("验证记录：", info["val_mask"].sum())
print("测试记录：", info["test_mask"].sum())
print("训练设备：", device)

if device.type == "cuda":
    print("GPU：", torch.cuda.get_device_name(0))

波形： (21388, 12, 1000)
标签： (21388, 5)
类别： ['NORM' 'MI' 'STTC' 'CD' 'HYP']
训练记录： 17084
验证记录： 2146
测试记录： 2158
训练设备： cuda
GPU： Tesla T4


In [3]:
from torch import nn
from torch.utils.data import Dataset, DataLoader

# 与 CNN 基线保持相同的数据加载方式
class ECGDataset(Dataset):
    def __init__(self, X, y, mask):
        self.X = X
        self.y = y
        self.indices = np.flatnonzero(mask)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        i = self.indices[index]
        waveform = torch.from_numpy(
            np.array(self.X[i], dtype=np.float32, copy=True)
        )
        label = torch.from_numpy(
            np.array(self.y[i], dtype=np.float32, copy=True)
        )
        return waveform, label

train_dataset = ECGDataset(X, y, info["train_mask"])
val_dataset = ECGDataset(X, y, info["val_mask"])

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    generator=torch.Generator().manual_seed(42),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        self.main = nn.Sequential(
            nn.Conv1d(
                in_channels, out_channels,
                kernel_size=5, stride=stride,
                padding=2, bias=False,
            ),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(),
            nn.Conv1d(
                out_channels, out_channels,
                kernel_size=5, padding=2, bias=False,
            ),
            nn.BatchNorm1d(out_channels),
        )

        # 当通道数或时间长度变化时，对齐旁路形状
        if in_channels != out_channels or stride != 1:
            self.shortcut = nn.Sequential(
                nn.Conv1d(
                    in_channels, out_channels,
                    kernel_size=1, stride=stride, bias=False,
                ),
                nn.BatchNorm1d(out_channels),
            )
        else:
            self.shortcut = nn.Identity()

        self.activation = nn.ReLU()

    def forward(self, x):
        return self.activation(self.main(x) + self.shortcut(x))

class ECGResNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv1d(12, 32, kernel_size=7, padding=3, bias=False),
            nn.BatchNorm1d(32),
            nn.ReLU(),

            ResidualBlock(32, 32),
            ResidualBlock(32, 64, stride=2),
            ResidualBlock(64, 128, stride=2),
            nn.AdaptiveAvgPool1d(1),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.2),
            nn.Linear(128, 5),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

# 创建模型前重新固定种子
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

model = ECGResNet().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# 检查一个验证批次，不更新模型
xb, yb = next(iter(val_loader))
model.eval()

with torch.no_grad():
    logits = model(xb.to(device))
    loss = criterion(logits, yb.to(device))

assert logits.shape == yb.shape == (len(xb), 5)
assert torch.isfinite(logits).all()
assert torch.isfinite(loss)

print("输入形状：", xb.shape)
print("输出形状：", logits.shape)
print("检查损失：", round(loss.item(), 4))
print("参数数量：", sum(p.numel() for p in model.parameters()))
print("ResNet 与数据加载器检查通过。")

输入形状： torch.Size([128, 12, 1000])
输出形状： torch.Size([128, 5])
检查损失： 0.7017
参数数量： 178757
ResNet 与数据加载器检查通过。


In [4]:
import time
import pandas as pd
from sklearn.metrics import roc_auc_score

save_dir = Path("/kaggle/working/resnet_baseline")
save_dir.mkdir(exist_ok=True)

epochs = 10
best_auc = -float("inf")
history = []

for epoch in range(1, epochs + 1):
    start = time.perf_counter()

    # 训练
    model.train()
    train_loss_sum = 0.0

    for xb, yb in train_loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = criterion(logits, yb)

        if not torch.isfinite(loss):
            raise RuntimeError("训练损失出现无效值")

        loss.backward()
        optimizer.step()
        train_loss_sum += loss.item() * xb.size(0)

    # 验证
    model.eval()
    val_loss_sum = 0.0
    probabilities = []
    targets = []

    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            logits = model(xb)
            loss = criterion(logits, yb)

            val_loss_sum += loss.item() * xb.size(0)
            probabilities.append(torch.sigmoid(logits).cpu().numpy())
            targets.append(yb.cpu().numpy())

    val_prob = np.concatenate(probabilities)
    val_true = np.concatenate(targets)

    train_loss = train_loss_sum / len(train_dataset)
    val_loss = val_loss_sum / len(val_dataset)
    val_auc = roc_auc_score(val_true, val_prob, average="macro")

    if val_auc > best_auc:
        best_auc = val_auc

        torch.save({
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "epoch": epoch,
            "val_macro_auc": float(val_auc),
            "classes": info["classes"].tolist(),
            "model_name": "ECGResNet",
            "seed": 42,
        }, save_dir / "best_model.pt")

        np.savez(
            save_dir / "best_validation_predictions.npz",
            probabilities=val_prob,
            targets=val_true,
            ecg_ids=info["ecg_ids"][info["val_mask"]],
        )

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_macro_auc": val_auc,
        "seconds": time.perf_counter() - start,
    })

    pd.DataFrame(history).to_csv(
        save_dir / "training_history.csv", index=False
    )

    print(
        f"第 {epoch:02d}/{epochs} 轮 | "
        f"训练损失 {train_loss:.4f} | "
        f"验证损失 {val_loss:.4f} | "
        f"验证 AUROC {val_auc:.4f} | "
        f"耗时 {history[-1]['seconds']:.1f} 秒",
        flush=True,
    )

print(f"\n训练结束，最佳验证 Macro AUROC：{best_auc:.4f}")
print("CNN 最佳验证 Macro AUROC：0.9174")
print("模型保存位置：", save_dir / "best_model.pt")

第 01/10 轮 | 训练损失 0.3788 | 验证损失 0.3475 | 验证 AUROC 0.8963 | 耗时 60.7 秒
第 02/10 轮 | 训练损失 0.3064 | 验证损失 0.3206 | 验证 AUROC 0.9047 | 耗时 4.4 秒
第 03/10 轮 | 训练损失 0.2850 | 验证损失 0.2941 | 验证 AUROC 0.9168 | 耗时 4.4 秒
第 04/10 轮 | 训练损失 0.2769 | 验证损失 0.2994 | 验证 AUROC 0.9203 | 耗时 4.4 秒
第 05/10 轮 | 训练损失 0.2672 | 验证损失 0.2888 | 验证 AUROC 0.9211 | 耗时 4.5 秒
第 06/10 轮 | 训练损失 0.2603 | 验证损失 0.2941 | 验证 AUROC 0.9209 | 耗时 4.5 秒
第 07/10 轮 | 训练损失 0.2557 | 验证损失 0.2832 | 验证 AUROC 0.9280 | 耗时 4.5 秒
第 08/10 轮 | 训练损失 0.2542 | 验证损失 0.2799 | 验证 AUROC 0.9275 | 耗时 4.6 秒
第 09/10 轮 | 训练损失 0.2492 | 验证损失 0.2926 | 验证 AUROC 0.9256 | 耗时 4.6 秒
第 10/10 轮 | 训练损失 0.2434 | 验证损失 0.2719 | 验证 AUROC 0.9294 | 耗时 4.7 秒

训练结束，最佳验证 Macro AUROC：0.9294
CNN 最佳验证 Macro AUROC：0.9174
模型保存位置： /kaggle/working/resnet_baseline/best_model.pt


In [5]:
from sklearn.metrics import (
    roc_auc_score,
    precision_recall_curve,
    precision_recall_fscore_support,
)

saved = np.load(save_dir / "best_validation_predictions.npz")
val_prob = saved["probabilities"]
val_true = saved["targets"]

# 固定阈值 0.5 的指标
p05, r05, f05, support = precision_recall_fscore_support(
    val_true,
    (val_prob >= 0.5).astype(int),
    average=None,
    zero_division=0,
)

# 每类分别选择验证 F1 最优阈值
thresholds = []

for i in range(len(info["classes"])):
    p, r, t = precision_recall_curve(
        val_true[:, i], val_prob[:, i]
    )
    scores = 2 * p[:-1] * r[:-1] / (
        p[:-1] + r[:-1] + 1e-12
    )
    thresholds.append(float(t[np.argmax(scores)]))

thresholds = np.array(thresholds)

p, r, f1, _ = precision_recall_fscore_support(
    val_true,
    (val_prob >= thresholds).astype(int),
    average=None,
    zero_division=0,
)

results = pd.DataFrame({
    "类别": info["classes"],
    "AUROC": roc_auc_score(val_true, val_prob, average=None),
    "F1（0.5）": f05,
    "选择的阈值": thresholds,
    "调阈值后 F1": f1,
    "调阈值后精确率": p,
    "调阈值后召回率": r,
    "阳性记录数": support,
})

display(results.round(4))
print(f"验证 Macro F1（0.5）：{f05.mean():.4f}")
print(f"验证 Macro F1（选择阈值后）：{f1.mean():.4f}")
print("CNN 对照：0.7055 / 0.7399")

np.save(save_dir / "thresholds.npy", thresholds)
results.to_csv(save_dir / "validation_metrics.csv", index=False)

,类别,AUROC,F1（0.5）,选择的阈值,调阈值后 F1,调阈值后精确率,调阈值后召回率,阳性记录数
0,NORM,0.9435,0.8544,0.4718,0.8560,0.8140,0.9026,955
1,MI,0.9269,0.7263,0.3745,0.7627,0.7759,0.7500,540
2,STTC,0.9333,0.7490,0.4146,0.7589,0.7303,0.7898,528
3,CD,0.9345,0.7638,0.4064,0.7712,0.7987,0.7455,495
4,HYP,0.9088,0.5154,0.3261,0.6207,0.6378,0.6045,268


验证 Macro F1（0.5）：0.7218
验证 Macro F1（选择阈值后）：0.7539
CNN 对照：0.7055 / 0.7399


In [6]:
# 恢复验证 AUROC 最佳轮次的模型
checkpoint = torch.load(
    save_dir / "best_model.pt",
    map_location=device,
    weights_only=True,
)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

thresholds = np.load(save_dir / "thresholds.npy")

test_dataset = ECGDataset(X, y, info["test_mask"])
test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

probabilities = []
targets = []

with torch.no_grad():
    for xb, yb in test_loader:
        logits = model(xb.to(device, non_blocking=True))
        probabilities.append(torch.sigmoid(logits).cpu().numpy())
        targets.append(yb.numpy())

test_prob = np.concatenate(probabilities)
test_true = np.concatenate(targets)

p, r, f1, support = precision_recall_fscore_support(
    test_true,
    (test_prob >= thresholds).astype(int),
    average=None,
    zero_division=0,
)

test_results = pd.DataFrame({
    "类别": info["classes"],
    "固定阈值": thresholds,
    "AUROC": roc_auc_score(test_true, test_prob, average=None),
    "精确率": p,
    "召回率": r,
    "F1": f1,
    "阳性记录数": support,
})

display(test_results.round(4))

test_auc = roc_auc_score(test_true, test_prob, average="macro")

_, _, f1_05, _ = precision_recall_fscore_support(
    test_true,
    (test_prob >= 0.5).astype(int),
    average=None,
    zero_division=0,
)

print(f"ResNet 测试 Macro AUROC：{test_auc:.4f}")
print(f"ResNet 测试 Macro F1（验证集阈值）：{f1.mean():.4f}")
print(f"ResNet 测试 Macro F1（0.5）：{f1_05.mean():.4f}")
print("CNN 测试对照：AUROC 0.9106 / F1 0.7277 / F1(0.5) 0.6978")

test_results.to_csv(save_dir / "test_metrics.csv", index=False)
np.savez(
    save_dir / "test_predictions.npz",
    probabilities=test_prob,
    targets=test_true,
    ecg_ids=info["ecg_ids"][info["test_mask"]],
    thresholds=thresholds,
)

,类别,固定阈值,AUROC,精确率,召回率,F1,阳性记录数
0,NORM,0.4718,0.9453,0.8083,0.9148,0.8583,963
1,MI,0.3745,0.9249,0.7514,0.7255,0.7382,550
2,STTC,0.4146,0.9371,0.7434,0.8119,0.7761,521
3,CD,0.4064,0.9192,0.7931,0.6956,0.7411,496
4,HYP,0.3261,0.9057,0.6349,0.6107,0.6226,262


ResNet 测试 Macro AUROC：0.9264
ResNet 测试 Macro F1（验证集阈值）：0.7473
ResNet 测试 Macro F1（0.5）：0.7334
CNN 测试对照：AUROC 0.9106 / F1 0.7277 / F1(0.5) 0.6978
